# SAÉ S2 : LABYRINTHES
**BUT Informatique de Reims - Saison 2026**

Binôme :
- Étudiant 1 : CHEVALLIOT Noa chev0161
- Étudiant 2 : CRAJKOWSKI Kris craj0001

Dépôt GitLab : `https://iut-info.univ-reims.fr/gitlab/chev0161/sae_labyrinthe.git`

---
> Ce notebook contient les **prototypes** de toutes les fonctions à implémenter.  
> Complétez chaque cellule marquée `# TODO` sans modifier les signatures.
> 
> Complétez le docstring en indiquant quel membre du binôme a compris et co-implémenté (**en retirant votre login, vous ne pourrez pas être interrogé à l'oral sur la fonction concernée**)
> 
> Effectuez des commits réguliers (au moins un par fonction).


## Imports

In [10]:
import random
import heapq
import time
from math import inf

---
## Classe `Maze`

La classe principale du projet. Le constructeur, `__str__`, `info`, `fill`, `empty`, `display` / `overlay` **vous sont fournis**.

Les méthodes marquées `😣 À faire` sont celles que vous devez implémenter.


In [31]:
class Maze:
    """
    Classe Labyrinthe
    Représentation sous forme de graphe non-orienté
    dont chaque sommet est une cellule (un tuple (l,c)) d'une grille,
    et dont la structure est représentée par un dictionnaire
      - clés : sommets
      - valeurs : ensemble des sommets voisins accessibles
    Attributs :
        height (int): hauteur de la grille du labyrinthe
        width  (int): largeur de la grille du labyrinthe
        neighbors (dict): dictionnaire des voisinages
        weight    (dict): dictionnaire des poids des déplacements (labyrinthe valué)
    """

    # ──────────────────────────────────────────────────────────────────────────
    # Constructeur et méthodes utilitaires (FOURNIES - NE PAS MODIFIER)
    # ──────────────────────────────────────────────────────────────────────────

    def __init__(self, height, width, empty=False):
        """
        Constructeur de labyrinthe à 'height' lignes et 'width' colonnes.
        Arguments:
            height (int) : nombre de lignes
            width  (int) : nombre de colonnes
            empty  (bool): True → labyrinthe sans murs ; False → tous les murs
        """
        self.weight = dict()
        self.height = height
        self.width  = width
        self.neighbors = dict()
        if empty:
            for i in range(self.height):
                for j in range(self.width):
                    self.neighbors[(i, j)] = set(
                        (ii, jj)
                        for ii, jj in [(i-1,j),(i+1,j),(i,j-1),(i,j+1)]
                        if 0 <= ii < self.height and 0 <= jj < self.width
                    )
        else:
            self.neighbors = {(i, j): set()
                              for i in range(height) for j in range(width)}

    def info(self):
        """Affichage des attributs (utile pour déboguer)."""
        txt  = "**Informations sur le labyrinthe**\n"
        txt += f"- Dimensions de la grille : {self.height} x {self.width}\n"
        txt += "- Voisinages :\n" + str(self.neighbors) + "\n"
        valid = True
        for c1 in {(i,j) for i in range(self.height) for j in range(self.width)}:
            for c2 in self.neighbors[c1]:
                if c1 not in self.neighbors[c2]:
                    valid = False
                    break
            else:
                continue
            break
        txt += "- Structure cohérente\n" if valid else f"- Structure incohérente : {c1} X {c2}\n"
        return txt

    def __str__(self):
        """Représentation ASCII du labyrinthe."""
        txt = "┏"
        for j in range(self.width-1):
            txt += "━━━┳"
        txt += "━━━┓\n┃"
        for j in range(self.width-1):
            txt += "   ┃" if (0,j+1) not in self.neighbors[(0,j)] else "    "
        txt += "   ┃\n"
        for i in range(self.height-1):
            txt += "┣"
            for j in range(self.width-1):
                txt += "━━━╋" if (i+1,j) not in self.neighbors[(i,j)] else "   ╋"
            txt += "━━━┫\n" if (i+1,self.width-1) not in self.neighbors[(i,self.width-1)] else "   ┫\n"
            txt += "┃"
            for j in range(self.width):
                txt += "   ┃" if (i+1,j+1) not in self.neighbors[(i+1,j)] else "    "
            txt += "\n"
        txt += "┗"
        for i in range(self.width-1):
            txt += "━━━┻"
        txt += "━━━┛\n"
        return txt

    def fill(self):
        """Remplit le labyrinthe avec tous les murs possibles."""
        for i in range(self.height):
            for j in range(self.width):
                if j+1 < self.width:  self.add_wall((i,j),(i,j+1))
                if i+1 < self.height: self.add_wall((i,j),(i+1,j))

    def empty(self):
        """Vide le labyrinthe de tous ses murs."""
        for i in range(self.height):
            for j in range(self.width):
                if j+1 < self.width:  self.remove_wall((i,j),(i,j+1))
                if i+1 < self.height: self.remove_wall((i,j),(i+1,j))

    def overlay(self, content=None):
        """Rendu ASCII avec contenu dans les cellules."""
        if content is None:
            content = {(i,j):' ' for i in range(self.height) for j in range(self.width)}
        else:
            new_content = {(i,j):' ' for i in range(self.height)
                           for j in range(self.width) if (i,j) not in content}
            content = {**content, **new_content}
        txt = "┏"
        for j in range(self.width-1): txt += "━━━┳"
        txt += "━━━┓\n┃"
        for j in range(self.width-1):
            txt += " "+content[(0,j)]+" ┃" if (0,j+1) not in self.neighbors[(0,j)] else " "+content[(0,j)]+"  "
        txt += " "+content[(0,self.width-1)]+" ┃\n"
        for i in range(self.height-1):
            txt += "┣"
            for j in range(self.width-1):
                txt += "━━━╋" if (i+1,j) not in self.neighbors[(i,j)] else "   ╋"
            txt += "━━━┫\n" if (i+1,self.width-1) not in self.neighbors[(i,self.width-1)] else "   ┫\n"
            txt += "┃"
            for j in range(self.width):
                txt += " "+content[(i+1,j)]+" ┃" if (i+1,j+1) not in self.neighbors[(i+1,j)] else " "+content[(i+1,j)]+"  "
            txt += "\n"
        txt += "┗"
        for i in range(self.width-1): txt += "━━━┻"
        txt += "━━━┛\n"
        return txt

    def display(self, content=None):
        """Alias de overlay."""
        return self.overlay(content)

    # ──────────────────────────────────────────────────────────────────────────
    # 1. Manipulation des murs  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def add_wall(self, c1, c2):
        """
        Ajout d'un mur entre deux cellules.
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        self.neighbors[c1].remove(c2)
        self.neighbors[c2].remove(c1)
        pass

    def remove_wall(self, c1, c2):
        """
        Suppression du mur entre deux cellules.
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        self.neighbors[c1].add(c2)
        self.neighbors[c2].add(c1)
        pass

    def get_all_walls(self):
        """
        Retourne la liste de tous les murs du labyrinthe
        sous la forme d'une liste de paires de cellules.
        Retour:
            list : liste de [c1, c2] représentant chaque mur
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        liste = []
        for c1 in self.neighbors : 
            c2 = (c1[0] + 1, c1[1])
            if c2 in self.neighbors and c2 not in self.neighbors[c1] :
                liste.append([c1, c2])
            c3 = (c1[0], c1[1] + 1)
            if c3 in self.neighbors and c3 not in self.neighbors[c1] :
                liste.append([c1, c3])
        return liste

    def get_reachable_cells(self, c): 
        """
        Liste des cellules accessibles depuis c
        (cellules contigües sans mur entre elles et c).
        Argument:
            c (tuple) : cellule de référence
        Retour:
            list : cellules accessibles
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """
        l_rep=[]
        l=self.get_contiguous_cells(c)
        for elt in l : 
            if c in self.neighbors[elt] :
                l_rep.append(elt)
        return l_rep

    def get_contiguous_cells(self, c):
        """
        Liste des cellules contigues à c
        (cellules adjacentes sur la grille indépendamment des murs ou de l'absence de mur).
        Argument:
            c (tuple) : cellule de référence
        Retour:
            list : cellules accessibles
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """
        l=[]
        i=c[0]
        j=c[1]
        if i-1>=0:
            l.append((i-1,j))
        if i+1<self.height:
            l.append((i+1,j))
        if j-1>=0:
            l.append((i,j-1))
        if j+1<self.width:
            l.append((i,j+1))
        return l


    # ──────────────────────────────────────────────────────────────────────────
    # 2. Génération  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    @classmethod
    def gen_btree(cls, h, w):
        
        """
        Génération par la méthode de l'arbre binaire.
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        laby = cls(h, w)
        for c1 in laby.neighbors : 
            c2 = (c1[0] + 1, c1[1])
            c3 = (c1[0], c1[1] + 1)
            if c2 in laby.neighbors and c3 in laby.neighbors : 
                numero = random.randint(1, 2)
                if numero == 1 : 
                    laby.remove_wall(c1, c2)
                else : 
                    laby.remove_wall(c1, c3)
            elif c3 not in laby.neighbors and c2 in laby.neighbors :
                laby.remove_wall(c1, c2)
            elif c2 not in laby.neighbors and c3 in laby.neighbors:
                laby.remove_wall(c1, c3)
        return laby

    @classmethod
    def gen_sidewinder(cls, h, w):
        """
        Génération par l'algorithme sidewinder.
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        laby = cls(h, w)

        for i in range(0, laby.height - 1):
            sequence = []
            for j in range(0, laby.width - 1):
                c1 = (i, j)
                sequence.append(c1)

                numero = random.randint(1, 2)

                if numero == 1:
                    c2 = (i, j + 1)
                    laby.remove_wall(c1, c2)
                else:
                    c3 = random.choice(sequence)
                    c4 = (c3[0] + 1, c3[1])
                    laby.remove_wall(c3, c4)
                    sequence = []

            der_cell = (i, laby.width - 1)
            sequence.append(der_cell)

            c5 = random.choice(sequence)
            c6 = (c5[0] + 1, c5[1])
            laby.remove_wall(c5, c6)

        i = laby.height - 1
        for j in range(0, laby.width - 1):
            c1 = (i, j)
            c2 = (i, j + 1)
            laby.remove_wall(c1, c2)

        return laby

    @classmethod
    def gen_fusion(cls, h, w):
        """
        Génération par fusion de chemins (variante de Kruskal).
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """

        laby=Maze(h,w)

        label={}
        num=1
        for elt in laby.neighbors:
            label[elt]=num
            num+=1

        l_mur=laby.get_all_walls()
        random.shuffle(l_mur)

        for mur in l_mur:
            if label[mur[0]]!=label[mur[1]]:
                laby.remove_wall(mur[0],mur[1])

                lab=label[mur[1]] #retiens le label de la cellule 2
                for c in label:
                    if label[c]==lab:
                        label[c]=label[mur[0]]

        return laby

    @classmethod
    def gen_exploration(cls, h, w):
        """
        Génération par exploration exhaustive (DFS aléatoire).
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """
        laby=Maze(h,w)
        x=random.randint(0,h-1)
        y=random.randint(0,w-1)
        c_debut=(x,y)

        l_marque=[c_debut]
        pile=[c_debut]

        while pile!=[]:
            cell=pile[-1]
            pile.pop(-1)

            l_contigue=laby.get_contiguous_cells(cell)
            contigue_visite=[]

            for cell_contigue in l_contigue:
                if cell_contigue not in l_marque:
                    contigue_visite.append(cell_contigue)

            if contigue_visite!=[]:
                pile.append(cell)
                cell_alea=random.choice(contigue_visite)
                laby.remove_wall(cell,cell_alea)
                l_marque.append(cell_alea)
                pile.append(cell_alea)

        return laby

    @classmethod
    def gen_wilson(cls, h, w):
        """
        Génération par l'algorithme de Wilson (marche aléatoire sans boucle).
        Arguments:
            h (int) : nombre de lignes
            w (int) : nombre de colonnes
        Retour:
            Maze : labyrinthe parfait généré
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        laby = cls(h, w)

        dico_marque = {}
        liste_non_m = []

        for case in laby.neighbors:
            dico_marque[case] = False
            liste_non_m.append(case)

        # Choisir une cellule au hasard sur la grille et la marquer
        cellule_dep = random.choice(liste_non_m)
        dico_marque[cellule_dep] = True
        liste_non_m.remove(cellule_dep)

        # Tant qu’il reste des cellules non marquées :
        while liste_non_m:
            
            # Choisir une cellule de départ au hasard, parmi les cellules non marquées
            c = random.choice(liste_non_m)
            chemin = [c]

            # Effectuer une marche aléatoire jusqu’à ce qu’une cellule marquée soit atteinte
            while not dico_marque[c]:
                voisins = laby.get_contiguous_cells(c)
                c = random.choice(voisins)

                if c in chemin :
                    index = chemin.index(c)
                    chemin = chemin[:index+1] # Prendre tout le chemin du début jusqu'a l'indice correspondant au début
                    # de la boucle
                else:
                    chemin.append(c)

            # Marquer chaque cellule du chemin, et casser tous les murs rencontrés, jusqu’à la cellule marquée
            for i in range(len(chemin) - 1):
                laby.remove_wall(chemin[i], chemin[i+1])

                if not dico_marque[chemin[i]]:
                    dico_marque[chemin[i]] = True
                    liste_non_m.remove(chemin[i])

            if not dico_marque[chemin[-1]]:
                dico_marque[chemin[-1]] = True
                liste_non_m.remove(chemin[-1])

        return laby

    # ──────────────────────────────────────────────────────────────────────────
    # 3. Résolution  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def solve_dfs(self, start, stop):
        """
        Résolution par parcours en profondeur (DFS).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de start à stop (liste de cellules)
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        
            # Parcours du graphe jusqu’à ce qu’on trouve A
        # Initialisation
        chemin = []

        pile = [start]
        predecesseur = {}
        predecesseur[start] = start
        marque = set([start])

        # Tant qu’il reste des cellules non-marquées
        save = False 
        
        while pile and not save:
            c = pile.pop()

            if c != stop:
                for voisin in self.get_reachable_cells(c):
                    if voisin not in marque:
                        marque.add(voisin)
                        pile.append(voisin)
                        predecesseur[voisin] = c
            else : 
                save = True


            #Construis le chemin par rapport au dictionnaire des predecesseurs
        # Initialiser c au stop
        c = stop
            
        # Tant que c n'est pas le départ :
        while c != start :
            chemin.append(c)
            c = predecesseur[c]
            
        # Ajouter le départ au chemin
        chemin.append(start)
            
        return chemin
            
                

    def solve_bfs(self, start, stop):
        """
        Résolution par parcours en largeur (BFS).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de start à stop (liste de cellules)
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
            # On parcours le graphe jusqu'à avoir A
        # Initialisation
        chemin = []

        file = [start]
        predecesseur = {}
        predecesseur[start] = start
        marque = set([start])

        # Tant qu’il reste des cellules non-marquées
        save = False 
        
        while file and not save:
            c = file.pop(0)

            if c != stop:
                for voisin in self.get_reachable_cells(c):
                    if voisin not in marque:
                        marque.add(voisin)
                        file.append(voisin)
                        predecesseur[voisin] = c
            else : 
                save = True


            # Construis le chemin par rapport au dictionnaire des predecesseurs
        # Initialiser c à A
        c = stop
            
        # Tant que c n'est pas le départ :
        while c != start :
            chemin.append(c)
            c = predecesseur[c]
            
        # Ajouter le départ au chemin
        chemin.append(start)
            
        return chemin

    def solve_rhr(self, start, stop):
        """
        Résolution par la règle de la main droite (right-hand rule).
        L'algorithme longe les murs en gardant toujours un contact
        avec le mur de droite.
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin trouvé (peut contenir des cycles)
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """

        cell=start #cell c'est la cellule où on est 
        chemin=[cell]
        directions=[(0,1),(-1,0),(0,-1),(1,0)]  # c'est les valeurs à ajouter au coord pour les cellules autour,  droite, haut, gauche, bas
        ind_dir=0 #on commence à droite

        while cell!=stop:

            reachable=self.get_reachable_cells(cell)
            trouve=False
            i=0

            #on test toutes les directions dans le sens des aiguilles d'une montre si on trouve la prochaone cellule accessible
            while not trouve and i<4:

                new_dir=(ind_dir+i-1)%4   #on ajoute la direction là où la cellule peut être (i) -1 vu qu'on veut tourner sur notre droite
                pro_cell=(cell[0] + directions[new_dir][0],cell[1] + directions[new_dir][1])

                if pro_cell in reachable:
                    ind_dir=new_dir
                    cell=pro_cell
                    chemin.append(cell)
                    trouve=True 

                i+=1
                
        return chemin

    # ──────────────────────────────────────────────────────────────────────────
    # 4. Métriques  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────


    def get_tree(self, start):
        """
        Extraction de l'arborescence de racine 'start' à l'aide d'un parcours.
        Arguments:
            start : cellule de départ
        Retour:
            dict : dictionnaire dont les clés sont les cellules du labyrinthe 
                   et les valeurs sont les prédécesseurs respectifs de ces cellules
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        pred = {}
        pile = [start]
        
        while pile :
            cellule = pile.pop()
            if cellule not in pred.keys() : 
                pred[cellule] = self.get_reachable_cells(cellule)
            for voisin in pred[cellule] :
                if voisin not in pred.keys() :
                     pile.append(voisin)
        
        return pred


    def distance_geo(self, c1, c2):
        """
        Distance géodésique entre c1 et c2 :
        nombre minimal de déplacements sur le graphe.
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Retour:
            int : longueur du chemin minimal
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        return len(self.solve_dfs(c1, c2))

    def distance_man(self, c1, c2):
        """
        Distance de Manhattan entre c1 et c2 sur la grille
        (sans tenir compte des murs).
        Arguments:
            c1 (tuple) : première cellule
            c2 (tuple) : deuxième cellule
        Retour:
            int : distance de Manhattan
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        return abs(c1[0] - c2[0]) + abs(c1[1] - c2[1])
        

    def tortuosity(self, start, stop):
        """
        Rapport entre la distance géodésique et la distance de Manhattan.
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            float : tortuosité (>= 1.0)
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """
        return self.distance_geo(start, stop) / self.distance_man(start, stop)

    def worst_path_len(self, start, stop):
        """
        Longueur du plus long chemin menant du départ à une impasse (feuille).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            int : longueur maximale d'une branche depuis start
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """
        l_impasse=[elt for elt in self.neighbors if len(self.get_reachable_cells(elt))==1 and elt!=stop]
        maxi=0
        for impasse in l_impasse:
            if maxi<self.distance_geo(start,impasse):
                maxi=self.distance_geo(start,impasse)
        return maxi
        
    def dead_end_number(self):
        """
        Nombre de culs-de-sac (cellules de degré 1).
        Retour:
            int : nombre de culs-de-sac
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        compteur = 0
        for cellule in self.neighbors : 
            if len(self.get_reachable_cells(cellule)) == 1 :
                compteur += 1
            
        return compteur

    def dead_end_rate(self):
        """
        Proportion de culs-de-sac par rapport au nombre total de cellules.
        Retour:
            float : taux de culs-de-sac
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        nb_cases = self.height * self.width
        return self.dead_end_number() / nb_cases

    def linearity_index(self):
        """
        Proportion de cellules de degré 2 (couloirs droits).
        Retour:
            float : index de linéarité
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """
        cpt=0
        for cell in self.neighbors : 
            if len(self.get_reachable_cells(cell))==2:
                cpt+=1
        return cpt/(self.height*self.width)

    # ──────────────────────────────────────────────────────────────────────────
    # 5. Labyrinthes imparfaits et valués  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def remove_random_wall(self):
        """
        Supprime un mur au hasard dans le labyrinthe
        (crée un passage entre deux cellules adjacentes non encore reliées).
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        mur = random.choice(self.get_all_walls())
        self.remove_wall(mur[0], mur[1])

    def add_random_weights(self, max_weight):
        """
        Ajoute un poids aléatoire (entre 1 et max_weight) à chaque
        déplacement possible. Les poids sont orientés : c1→c2 peut
        différer de c2→c1.
        Argument:
            max_weight (int) : poids maximal possible
        Membre(s) du binôme concerné(s) : chev0161, craj0001
        """
        for case in self.neighbors : 
            for voisin in self.get_reachable_cells(case) : 
                self.weight[(case, voisin)] = random.randint(1, max_weight)


    # ──────────────────────────────────────────────────────────────────────────
    # 6. Résolution de labyrinthes valués  😣 À FAIRE
    # ──────────────────────────────────────────────────────────────────────────

    def solve_dijkstra(self, start, stop):
        """
        Chemin de coût minimal - Dijkstra version naïve (sans tas).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de coût minimal de start à stop
        Membre(s) du binôme concerné(s) : craj0001, chev0161
        """
        
        chemin=[]

        cells=[]  #liste de toutes les cellules
        for cellule in self.neighbors:
            cells.append(cellule)

        M=[]
        for cell in self.neighbors:
            ligne=[]
            for cell2 in self.neighbors:
                if cell2 in self.get_reachable_cells(cell):
                    ligne.append(self.weight[(cell,cell2)])
                else:
                    ligne.append(inf)
            M.append(ligne)

        Dist=[inf for elt in self.neighbors]

        Dist[cells.index(start)]=0

        marquage=[False for elt in self.neighbors]

        avant=[None for _ in cells] #servira à contenir l'indice de la cellule d'où on vient 
        # (Ex: avant[6]=4 ça veut dire que pour aller à la cellule 6 on est passé par 4)

        while False in marquage: 

            mini=inf
            for i in range(len(Dist)):
                if mini>Dist[i] and marquage[i]==False:
                    mini=Dist[i]
                    ind=i 

            marquage[ind]=True

            for j in range(len(marquage)):
                if marquage[j]==False:
                    if Dist[ind]+M[ind][j]<Dist[j]:
                        Dist[j]=Dist[ind]+M[ind][j]
                        avant[j]=ind

        c=cells.index(stop)

        while c is not None:   #avant[start] toujours = None
            chemin.append(cells[c])
            c=avant[c]

        chemin.reverse()

        return chemin
    
    def to_html(self, content=None, cell_size=32, start=None, stop=None, show_weights=False):
        """
        Rendu HTML/SVG du labyrinthe.
        Arguments:
            content      (dict) : dictionnaire {cellule: caractère} pour annoter les cellules
            cell_size    (int)  : taille en pixels d'une cellule
            start        (tuple): cellule de départ (affichée en vert)
            stop         (tuple): cellule d'arrivée (affichée en rouge)
            show_weights (bool) : si True et que le labyrinthe est valué (self.weight non vide),
                                  colore les passages selon l'intensité du poids de l'arête
                                  (bleu pâle = poids faible, orange vif = poids élevé)
        Retour:
            string : code HTML
        """
        from IPython.display import display, HTML

        W = self.width  * cell_size
        H = self.height * cell_size
        wall_w = 2        # épaisseur des murs intérieurs
        border_w = 3      # épaisseur du bord extérieur
        wall_color   = "#2d2d2d"
        path_color   = "#a78bfa"   # violet pour le chemin
        start_color  = "#4ade80"   # vert
        stop_color   = "#f87171"   # rouge
        cell_bg      = "#fafafa"
        passage_bg   = "#ffffff"

        # --- Calcul des couleurs d'intensité des poids ---
        # Pour chaque passage ouvert (arête), on colore un fin rectangle
        # dans le couloir entre les deux cellules selon le poids de l'arête.
        # Palette : bleu (#93c5fd) → poids min  →  orange (#fb923c) → poids max
        def _weight_color(w, w_min, w_max):
            """Interpolation linéaire bleu → orange selon l'intensité du poids."""
            if w_max == w_min:
                t = 0.5
            else:
                t = (w - w_min) / (w_max - w_min)
            # bleu clair  (147, 197, 253)  →  orange vif  (251, 146,  60)
            r = int(147 + t * (251 - 147))
            g = int(197 + t * (146 - 197))
            b = int(253 + t * ( 60 - 253))
            return f"#{r:02x}{g:02x}{b:02x}"

        has_weights = show_weights and bool(self.weight)
        if has_weights:
            all_w = list(self.weight.values())
            w_min, w_max = min(all_w), max(all_w)

        svg = f'<svg xmlns="http://www.w3.org/2000/svg" width="{W + border_w}" height="{H + border_w}" style="display:block; margin:8px 0;">\n'

        # Fond général
        svg += f'  <rect width="{W + border_w}" height="{H + border_w}" fill="{cell_bg}"/>\n'

        # ── Passages colorés selon le poids (dessinés AVANT les cellules) ──
        if has_weights:
            passage_thick = max(4, cell_size // 4)  # épaisseur du bandeau coloré
            for i in range(self.height):
                for j in range(self.width):
                    x = j * cell_size + border_w // 2
                    y = i * cell_size + border_w // 2
                    # Passage EST : arête (i,j)→(i,j+1)
                    if j + 1 < self.width and (i, j+1) in self.neighbors[(i, j)]:
                        w = self.weight.get(((i,j),(i,j+1)),
                            self.weight.get(((i,j+1),(i,j)), w_min))
                        col = _weight_color(w, w_min, w_max)
                        rx = x + cell_size - passage_thick // 2
                        ry = y + (cell_size - passage_thick) // 2
                        svg += (f'  <rect x="{rx}" y="{ry}" '
                                f'width="{passage_thick}" height="{passage_thick}" '
                                f'fill="{col}" opacity="0.85"/>\n')
                    # Passage SUD : arête (i,j)→(i+1,j)
                    if i + 1 < self.height and (i+1, j) in self.neighbors[(i, j)]:
                        w = self.weight.get(((i,j),(i+1,j)),
                            self.weight.get(((i+1,j),(i,j)), w_min))
                        col = _weight_color(w, w_min, w_max)
                        rx = x + (cell_size - passage_thick) // 2
                        ry = y + cell_size - passage_thick // 2
                        svg += (f'  <rect x="{rx}" y="{ry}" '
                                f'width="{passage_thick}" height="{passage_thick}" '
                                f'fill="{col}" opacity="0.85"/>\n')

        # Cellules colorées (start / stop / chemin)
        if content:
            for (i, j), char in content.items():
                x = j * cell_size + border_w // 2
                y = i * cell_size + border_w // 2
                if (i, j) == start:
                    color = start_color
                elif (i, j) == stop:
                    color = stop_color
                elif char.strip():
                    color = path_color
                else:
                    color = passage_bg
                svg += f'  <rect x="{x}" y="{y}" width="{cell_size}" height="{cell_size}" fill="{color}" opacity="0.6"/>\n'

        if start and (not content or start not in content):
            x = start[1] * cell_size + border_w // 2
            y = start[0] * cell_size + border_w // 2
            svg += f'  <rect x="{x}" y="{y}" width="{cell_size}" height="{cell_size}" fill="{start_color}" opacity="0.7"/>\n'

        if stop and (not content or stop not in content):
            x = stop[1] * cell_size + border_w // 2
            y = stop[0] * cell_size + border_w // 2
            svg += f'  <rect x="{x}" y="{y}" width="{cell_size}" height="{cell_size}" fill="{stop_color}" opacity="0.7"/>\n'

        # Murs intérieurs
        for i in range(self.height):
            for j in range(self.width):
                x = j * cell_size + border_w // 2
                y = i * cell_size + border_w // 2
                # Mur EST
                if j + 1 < self.width and (i, j+1) not in self.neighbors[(i, j)]:
                    svg += f'  <line x1="{x + cell_size}" y1="{y}" x2="{x + cell_size}" y2="{y + cell_size}" stroke="{wall_color}" stroke-width="{wall_w}" stroke-linecap="square"/>\n'
                # Mur SUD
                if i + 1 < self.height and (i+1, j) not in self.neighbors[(i, j)]:
                    svg += f'  <line x1="{x}" y1="{y + cell_size}" x2="{x + cell_size}" y2="{y + cell_size}" stroke="{wall_color}" stroke-width="{wall_w}" stroke-linecap="square"/>\n'

        # Bordure extérieure
        svg += f'  <rect x="{border_w//2}" y="{border_w//2}" width="{W}" height="{H}" fill="none" stroke="{wall_color}" stroke-width="{border_w}"/>\n'

        # Labels dans les cellules
        if content:
            font_size = max(10, cell_size // 3)
            for (i, j), char in content.items():
                if char.strip():
                    cx = j * cell_size + cell_size // 2 + border_w // 2
                    cy = i * cell_size + cell_size // 2 + border_w // 2 + font_size // 3
                    svg += f'  <text x="{cx}" y="{cy}" text-anchor="middle" font-size="{font_size}" font-family="monospace" font-weight="bold" fill="#1e1e1e">{char}</text>\n'

        # Légende des poids (si activée)
        if has_weights:
            leg_x = border_w // 2
            leg_w = min(W, 200)
            leg_h = 10
            steps = 20
            step_w = leg_w // steps
            svg_leg = f'<svg xmlns="http://www.w3.org/2000/svg" width="{W + border_w}" height="30" style="display:block; margin:0;">\n'
            for k in range(steps):
                t_val = k / (steps - 1)
                w_val = w_min + t_val * (w_max - w_min)
                col = _weight_color(w_val, w_min, w_max)
                svg_leg += f'  <rect x="{leg_x + k * step_w}" y="2" width="{step_w + 1}" height="{leg_h}" fill="{col}"/>\n'
            svg_leg += f'  <text x="{leg_x}" y="{leg_h + 14}" font-size="9" font-family="monospace" fill="#555">{w_min}</text>\n'
            svg_leg += f'  <text x="{leg_x + leg_w}" y="{leg_h + 14}" font-size="9" font-family="monospace" fill="#555" text-anchor="end">{w_max}</text>\n'
            svg_leg += f'  <text x="{leg_x + leg_w // 2}" y="{leg_h + 14}" font-size="9" font-family="monospace" fill="#555" text-anchor="middle">poids</text>\n'
            svg_leg += '</svg>'
            svg += '</svg>'
            display(HTML(svg + svg_leg))
            return

        svg += '</svg>'
        display(HTML(svg))

    def solve_dijkstra_whq(self, start, stop):
        """
        [BONUS]
        Chemin de coût minimal - Dijkstra avec liste à priorité (heapq).
        Arguments:
            start (tuple) : cellule de départ
            stop  (tuple) : cellule d'arrivée
        Retour:
            list : chemin de coût minimal de start à stop
        Membre(s) du binôme concerné(s) : chev0161
        """
        pass
        


---
## Section 1 - Manipulation des murs

### Tests de `remove_wall`


In [32]:
laby = Maze(5, 5)
print("Labyrinthe plein :")
print(laby)

laby.remove_wall((0, 0), (0, 1))
print("Après remove_wall((0,0),(0,1)) :")
print(laby)

Labyrinthe plein :
┏━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┛

Après remove_wall((0,0),(0,1)) :
┏━━━┳━━━┳━━━┳━━━┳━━━┓
┃       ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┣━━━╋━━━╋━━━╋━━━╋━━━┫
┃   ┃   ┃   ┃   ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┛



### Tests de `get_all_walls`

In [33]:
laby2 = Maze(3, 3, empty=True)
laby2.add_wall((0, 0), (0, 1))
laby2.add_wall((1, 1), (1, 2))
print("Murs du labyrinthe :")
print(laby2.get_all_walls())


Murs du labyrinthe :
[[(0, 0), (0, 1)], [(1, 1), (1, 2)]]


### Tests de `get_reachable_cells`

In [34]:
laby3 = Maze(4, 4)
laby3.neighbors = {
    (0,0): {(1,0)},
    (0,1): {(0,2),(1,1)},
    (0,2): {(0,1),(0,3)},
    (0,3): {(0,2),(1,3)},
    (1,0): {(2,0),(0,0)},
    (1,1): {(0,1),(1,2)},
    (1,2): {(1,1),(2,2)},
    (1,3): {(2,3),(0,3)},
    (2,0): {(1,0),(2,1),(3,0)},
    (2,1): {(2,0),(2,2)},
    (2,2): {(1,2),(2,1)},
    (2,3): {(3,3),(1,3)},
    (3,0): {(3,1),(2,0)},
    (3,1): {(3,2),(3,0)},
    (3,2): {(3,1)},
    (3,3): {(2,3)}
}
print("Cellules contigues de (2,0) :", laby3.get_contiguous_cells((2,0)))
print("Cellules accessibles de (2,0) :", laby3.get_reachable_cells((2,0)))


Cellules contigues de (2,0) : [(1, 0), (3, 0), (2, 1)]
Cellules accessibles de (2,0) : [(1, 0), (3, 0), (2, 1)]


---
## Section 2 - Génération

### `gen_btree`


In [35]:
laby_btree = Maze.gen_btree(6, 6)
print(laby_btree)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃           ┃   ┃   ┃   ┃
┣━━━╋━━━╋   ╋   ╋   ╋   ┫
┃   ┃           ┃   ┃   ┃
┣   ╋━━━╋━━━╋   ╋   ╋   ┫
┃   ┃   ┃   ┃   ┃   ┃   ┃
┣   ╋   ╋   ╋   ╋   ╋   ┫
┃   ┃           ┃       ┃
┣   ╋━━━╋━━━╋   ╋━━━╋   ┫
┃   ┃           ┃   ┃   ┃
┣   ╋━━━╋━━━╋   ╋   ╋   ┫
┃                       ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_sidewinder`

In [36]:
laby_sw = Maze.gen_sidewinder(6, 6)
print(laby_sw)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃   ┃   ┃   ┃   ┃       ┃
┣   ╋   ╋   ╋   ╋━━━╋   ┫
┃   ┃   ┃           ┃   ┃
┣   ╋   ╋━━━╋   ╋━━━╋   ┫
┃           ┃   ┃       ┃
┣   ╋━━━╋━━━╋   ╋   ╋━━━┫
┃       ┃       ┃       ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━┫
┃   ┃                   ┃
┣   ╋   ╋━━━╋━━━╋━━━╋━━━┫
┃                       ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_fusion`

In [37]:
laby_fusion = Maze.gen_fusion(10, 10)
print(laby_fusion)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃           ┃   ┃   ┃   ┃       ┃   ┃   ┃
┣   ╋   ╋   ╋   ╋   ╋   ╋━━━╋   ╋   ╋   ┫
┃   ┃   ┃       ┃               ┃       ┃
┣   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━┫
┃       ┃   ┃           ┃   ┃           ┃
┣━━━╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━╋━━━┫
┃           ┃           ┃   ┃           ┃
┣━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋   ╋   ╋   ┫
┃               ┃   ┃       ┃   ┃   ┃   ┃
┣━━━╋   ╋   ╋   ╋   ╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃   ┃   ┃   ┃       ┃           ┃
┣   ╋   ╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋━━━┫
┃   ┃   ┃           ┃   ┃   ┃           ┃
┣━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋   ╋━━━┫
┃   ┃       ┃   ┃               ┃       ┃
┣   ╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃   ┃   ┃   ┃       ┃   ┃   ┃       ┃   ┃
┣   ╋   ╋   ╋   ╋   ╋   ╋━━━╋━━━╋━━━╋   ┫
┃       ┃       ┃   ┃               ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_exploration`

In [38]:
laby_exp = Maze.gen_exploration(10, 10)
print(laby_exp)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃               ┃                       ┃
┣━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ┫
┃           ┃           ┃   ┃       ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ┫
┃   ┃           ┃   ┃   ┃   ┃           ┃
┣   ╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋━━━┫
┃           ┃   ┃   ┃   ┃               ┃
┣━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃           ┃   ┃   ┃   ┃           ┃   ┃
┣   ╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ╋━━━╋   ┫
┃   ┃           ┃       ┃   ┃   ┃       ┃
┣   ╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ╋   ┫
┃   ┃           ┃   ┃           ┃   ┃   ┃
┣   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━┫
┃       ┃   ┃       ┃           ┃       ┃
┣━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ┫
┃           ┃   ┃       ┃       ┃       ┃
┣   ╋━━━╋━━━╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃                       ┃           ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `gen_wilson`

In [39]:
laby_wilson = Maze.gen_wilson(10, 10)
print(laby_wilson)

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃               ┃       ┃               ┃
┣━━━╋   ╋━━━╋   ╋   ╋━━━╋   ╋━━━╋━━━╋   ┫
┃       ┃       ┃       ┃       ┃       ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━┫
┃   ┃       ┃           ┃   ┃   ┃       ┃
┣   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━┫
┃   ┃               ┃       ┃   ┃       ┃
┣   ╋━━━╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ╋━━━┫
┃   ┃       ┃       ┃       ┃   ┃       ┃
┣   ╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋━━━┫
┃   ┃       ┃           ┃   ┃   ┃       ┃
┣   ╋━━━╋   ╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ┫
┃   ┃   ┃       ┃               ┃       ┃
┣   ╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃       ┃   ┃   ┃   ┃   ┃   ┃       ┃   ┃
┣   ╋━━━╋   ╋━━━╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃           ┃       ┃   ┃   ┃   ┃       ┃
┣━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ╋   ╋   ┫
┃                           ┃       ┃   ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



---
## Section 3 - Résolution

### `solve_dfs`


In [40]:
laby = Maze.gen_fusion(10, 10)
solution = laby.solve_dfs((0, 0), (9, 9))
content = {c: '*' for c in solution}
content[(0, 0)] = 'D'
content[(9, 9)] = 'A'
print(laby.overlay(content))

┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D   *     ┃   ┃           ┃           ┃
┣━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━┫
┃     * ┃               ┃       ┃   ┃   ┃
┣━━━╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃   ┃ *     ┃   ┃   ┃   ┃           ┃   ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃     *             ┃           ┃       ┃
┣━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━┫
┃ *   * ┃       ┃       ┃               ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━┫
┃ *   *   *   *   *   *   * ┃   ┃       ┃
┣━━━╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ┫
┃       ┃       ┃   ┃   ┃ *   * ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━┫
┃   ┃       ┃           ┃   ┃ *   *     ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━┫
┃   ┃           ┃   ┃   ┃   ┃     *   * ┃
┣   ╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋━━━╋   ┫
┃       ┃       ┃               ┃     A ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `solve_bfs`

In [41]:
solution_bfs = laby.solve_bfs((0, 0), (9, 9))
content_bfs = {c: '*' for c in solution_bfs}
content_bfs[(0, 0)] = 'D'
content_bfs[(9, 9)] = 'A'
print(laby.overlay(content_bfs))


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D   *     ┃   ┃           ┃           ┃
┣━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━┫
┃     * ┃               ┃       ┃   ┃   ┃
┣━━━╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃   ┃ *     ┃   ┃   ┃   ┃           ┃   ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃     *             ┃           ┃       ┃
┣━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━┫
┃ *   * ┃       ┃       ┃               ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━┫
┃ *   *   *   *   *   *   * ┃   ┃       ┃
┣━━━╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ┫
┃       ┃       ┃   ┃   ┃ *   * ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━┫
┃   ┃       ┃           ┃   ┃ *   *     ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━┫
┃   ┃           ┃   ┃   ┃   ┃     *   * ┃
┣   ╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋━━━╋   ┫
┃       ┃       ┃               ┃     A ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `solve_rhr` - main droite

In [42]:
solution_rhr = laby.solve_rhr((0, 0), (9, 9))
content_rhr = {c: '*' for c in solution_rhr}
content_rhr[(0, 0)] = 'D'
content_rhr[(9, 9)] = 'A'
print(laby.overlay(content_rhr))


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D   *     ┃   ┃           ┃           ┃
┣━━━╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ╋━━━┫
┃ *   * ┃               ┃       ┃   ┃   ┃
┣━━━╋   ╋━━━╋   ╋   ╋   ╋   ╋━━━╋━━━╋   ┫
┃ * ┃ *     ┃   ┃   ┃   ┃           ┃   ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋   ┫
┃ *   *             ┃           ┃       ┃
┣━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━┫
┃ *   * ┃       ┃       ┃               ┃
┣   ╋━━━╋   ╋━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━┫
┃ *   *   *   *   *   *   * ┃   ┃       ┃
┣━━━╋   ╋   ╋━━━╋   ╋━━━╋   ╋━━━╋   ╋   ┫
┃ *   * ┃ *   * ┃ * ┃ * ┃ *   * ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ╋━━━┫
┃ * ┃ *   * ┃ *   *   * ┃ * ┃ *   *     ┃
┣   ╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋   ╋━━━┫
┃ * ┃ *   *   * ┃ * ┃ * ┃ * ┃ *   *   * ┃
┣   ╋   ╋   ╋━━━╋   ╋   ╋━━━╋   ╋━━━╋   ┫
┃ *   * ┃ *   * ┃ *   *   *   * ┃     A ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



---
## Section 4 - Métriques

### Distances


In [43]:
laby_m = Maze.gen_fusion(5, 5)
laby_m.get_tree((3, 3))

{(3, 3): [(4, 3)],
 (4, 3): [(3, 3), (4, 2), (4, 4)],
 (4, 4): [(3, 4), (4, 3)],
 (3, 4): [(4, 4)],
 (4, 2): [(3, 2), (4, 1), (4, 3)],
 (4, 1): [(4, 2)],
 (3, 2): [(2, 2), (4, 2), (3, 1)],
 (3, 1): [(3, 0), (3, 2)],
 (3, 0): [(2, 0), (4, 0), (3, 1)],
 (4, 0): [(3, 0)],
 (2, 0): [(3, 0), (2, 1)],
 (2, 1): [(1, 1), (2, 0)],
 (1, 1): [(2, 1), (1, 0)],
 (1, 0): [(1, 1)],
 (2, 2): [(3, 2), (2, 3)],
 (2, 3): [(1, 3), (2, 2), (2, 4)],
 (2, 4): [(1, 4), (2, 3)],
 (1, 4): [(0, 4), (2, 4)],
 (0, 4): [(1, 4), (0, 3)],
 (0, 3): [(0, 2), (0, 4)],
 (0, 2): [(1, 2), (0, 1), (0, 3)],
 (0, 1): [(0, 0), (0, 2)],
 (0, 0): [(0, 1)],
 (1, 2): [(0, 2)],
 (1, 3): [(2, 3)]}

In [44]:
laby_m = Maze.gen_fusion(10, 10)
c1, c2 = (0, 0), (9, 9)

print(f"Distance géodésique  {c1} → {c2} : {laby_m.distance_geo(c1, c2)}")
print(f"Distance de Manhattan {c1} → {c2} : {laby_m.distance_man(c1, c2)}")

Distance géodésique  (0, 0) → (9, 9) : 19
Distance de Manhattan (0, 0) → (9, 9) : 18


### Tortuosité et longueur pire chemin

In [45]:
print(f"Tortuosité          : {laby_m.tortuosity(c1, c2):.3f}")
print(f"Longueur pire chemin: {laby_m.worst_path_len(c1, c2)}")


Tortuosité          : 1.056
Longueur pire chemin: 27


### Culs-de-sac et linéarité

In [46]:
print(f"Nombre de culs-de-sac : {laby_m.dead_end_number()}")
print(f"Taux de culs-de-sac   : {laby_m.dead_end_rate():.3f}")
print(f"Index de linéarité    : {laby_m.linearity_index():.3f}")


Nombre de culs-de-sac : 29
Taux de culs-de-sac   : 0.290
Index de linéarité    : 0.490


---
## Section 5 - Labyrinthes imparfaits et valués

### `remove_random_wall`


In [47]:
laby_imp = Maze.gen_fusion(8, 8)
print("Avant suppression d'un mur aléatoire :")
print(laby_imp.info())
laby_imp.remove_random_wall()
print("Après suppression d'un mur aléatoire :")
print(laby_imp.info())


Avant suppression d'un mur aléatoire :
**Informations sur le labyrinthe**
- Dimensions de la grille : 8 x 8
- Voisinages :
{(0, 0): {(0, 1), (1, 0)}, (0, 1): {(0, 2), (0, 0)}, (0, 2): {(0, 1), (1, 2), (0, 3)}, (0, 3): {(0, 2), (1, 3), (0, 4)}, (0, 4): {(1, 4), (0, 3), (0, 5)}, (0, 5): {(0, 4), (1, 5)}, (0, 6): {(0, 7), (1, 6)}, (0, 7): {(0, 6)}, (1, 0): {(2, 0), (0, 0)}, (1, 1): {(2, 1)}, (1, 2): {(0, 2)}, (1, 3): {(0, 3)}, (1, 4): {(0, 4)}, (1, 5): {(1, 6), (2, 5), (0, 5)}, (1, 6): {(1, 5), (0, 6)}, (1, 7): {(2, 7)}, (2, 0): {(1, 0), (2, 1), (3, 0)}, (2, 1): {(1, 1), (2, 0)}, (2, 2): {(2, 3), (3, 2)}, (2, 3): {(2, 4), (2, 2)}, (2, 4): {(2, 3)}, (2, 5): {(2, 6), (1, 5)}, (2, 6): {(2, 5), (3, 6)}, (2, 7): {(3, 7), (1, 7)}, (3, 0): {(2, 0)}, (3, 1): {(3, 2), (4, 1)}, (3, 2): {(3, 1), (3, 3), (2, 2)}, (3, 3): {(3, 2)}, (3, 4): {(4, 4), (3, 5)}, (3, 5): {(3, 4), (3, 6)}, (3, 6): {(3, 7), (4, 6), (2, 6), (3, 5)}, (3, 7): {(2, 7), (3, 6)}, (4, 0): {(5, 0)}, (4, 1): {(3, 1), (5, 1), (4, 2)}, 

### `add_random_weights` et degrés

In [48]:
laby_w = Maze.gen_fusion(8, 8)
laby_w.add_random_weights(max_weight=10)
print(f"Degré moyen  : {laby_w.average_degree():.3f}")
print(f"Degré médian : {laby_w.median_degree():.3f}")
print(f"Extrait des poids : {dict(list(laby_w.weight.items())[:5])}")

AttributeError: 'Maze' object has no attribute 'average_degree'

---
## Section 6 - Résolution de labyrinthes valués

### `solve_dijkstra` (version naïve)


In [49]:
laby_d = Maze.gen_fusion(10, 10)
laby_d.add_random_weights(max_weight=10)
start, stop = (0, 0), (9, 9)

path_dijk = laby_d.solve_dijkstra(start, stop)
content_d = {c: '*' for c in path_dijk}
content_d[start] = 'D'
content_d[stop]  = 'A'
print(laby_d.overlay(content_d))


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D         ┃                   ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ┫
┃ * ┃ *   *     ┃   ┃   ┃   ┃   ┃       ┃
┣   ╋   ╋   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋   ┫
┃ * ┃ * ┃ *   *         ┃           ┃   ┃
┣   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━┫
┃ *   *     ┃ *   *   *     ┃       ┃   ┃
┣━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ┫
┃       ┃           ┃ *   *   *   *     ┃
┣   ╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━┫
┃   ┃               ┃         *   *     ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃           ┃   ┃   ┃ *   * ┃   ┃
┣   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ┫
┃   ┃               ┃             *   * ┃
┣━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋   ┫
┃           ┃           ┃           ┃ * ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ┫
┃               ┃           ┃   ┃     A ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### `solve_dijkstra_whq` (avec heapq)

In [50]:
path_whq = laby_d.solve_dijkstra_whq(start, stop)
content_whq = {c: '*' for c in path_whq}
content_whq[start] = 'D'
content_whq[stop]  = 'A'
print(laby_d.overlay(content_whq))


┏━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┳━━━┓
┃ D         ┃                   ┃   ┃   ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋   ┫
┃ * ┃ *   *     ┃   ┃   ┃   ┃   ┃       ┃
┣   ╋   ╋   ╋━━━╋   ╋   ╋   ╋━━━╋   ╋   ┫
┃ * ┃ * ┃ *   *         ┃           ┃   ┃
┣   ╋   ╋━━━╋   ╋━━━╋━━━╋━━━╋━━━╋   ╋━━━┫
┃ *   *     ┃ *   *   *     ┃       ┃   ┃
┣━━━╋   ╋━━━╋━━━╋   ╋   ╋━━━╋━━━╋   ╋   ┫
┃       ┃           ┃ *   *   *   *     ┃
┣   ╋━━━╋━━━╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋━━━┫
┃   ┃               ┃         *   *     ┃
┣   ╋━━━╋   ╋━━━╋━━━╋━━━╋   ╋   ╋━━━╋━━━┫
┃       ┃           ┃   ┃   ┃ *   * ┃   ┃
┣   ╋━━━╋   ╋━━━╋━━━╋   ╋━━━╋━━━╋   ╋   ┫
┃   ┃               ┃             *   * ┃
┣━━━╋   ╋━━━╋━━━╋━━━╋━━━╋━━━╋   ╋━━━╋   ┫
┃           ┃           ┃           ┃ * ┃
┣   ╋━━━╋━━━╋━━━╋━━━╋   ╋   ╋   ╋━━━╋   ┫
┃               ┃           ┃   ┃     A ┃
┗━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┻━━━┛



### Comparaison des deux Dijkstra (temps d'exécution)

In [52]:
import time

def time_solver(solver, laby, start, stop, n=50):
    t0 = time.perf_counter()
    for _ in range(n):
        solver(start, stop)
    return (time.perf_counter() - t0) / n * 1000  # ms

laby_bench = Maze.gen_fusion(20, 20)
laby_bench.add_random_weights(max_weight=10)
s, e = (0, 0), (19, 19)

t_naive = time_solver(laby_bench.solve_dijkstra,     laby_bench, s, e)
t_heapq = time_solver(laby_bench.solve_dijkstra_whq, laby_bench, s, e)

print(f"Dijkstra naïf  : {t_naive:.3f} ms")
print(f"Dijkstra heapq : {t_heapq:.3f} ms")
print(f"Accélération   : ×{t_naive/t_heapq:.1f}")


KeyboardInterrupt: 

---
## Section 7 - Benchmark global des générateurs

Utiliser ce code une fois toutes les méthodes implémentées pour comparer les générateurs.


In [400]:
def benchmark(generators, h, w, n):
    """Calcule les métriques moyennes pour chaque générateur."""
    results = {}
    for gen in generators:
        name = gen.__name__
        data = {k: [] for k in ['tortuosity','worst_path_len','dead_end_rate',
                                 'linearity_index','distance_geo','dead_end_number']}
        for _ in range(n):
            laby = gen(h, w)
            s, e = (0, 0), (h-1, w-1)
            data['tortuosity'].append(laby.tortuosity(s, e))
            data['worst_path_len'].append(laby.worst_path_len(s, e))
            data['dead_end_rate'].append(laby.dead_end_rate())
            data['linearity_index'].append(laby.linearity_index())
            data['distance_geo'].append(laby.distance_geo(s, e))
            data['dead_end_number'].append(laby.dead_end_number())
        results[name] = {k: sum(v)/n for k, v in data.items()}
    # Affichage simple en tableau
    header = f"{'Générateur':<20} " + " ".join(f"{k:>16}" for k in data)
    print(header)
    print("-" * len(header))
    for name, vals in results.items():
        row = f"{name:<20} " + " ".join(f"{v:>16.3f}" for v in vals.values())
        print(row)

generators = [Maze.gen_btree, Maze.gen_sidewinder, Maze.gen_fusion,
              Maze.gen_exploration, Maze.gen_wilson]
benchmark(generators, h=12, w=12, n=20)


TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'